# 04c: Attention Patterns: Circuit Analysis

## Overview
Analyzes attention patterns under **mean ablation** (circuit mode).
Non-circuit edges have their outputs replaced with dataset-mean activations,
isolating the computational path through the ACDC-discovered circuit.

## Key Questions
1. How does ablation change attention patterns from the prediction position?
2. Are head role classifications preserved under circuit ablation?
3. Does attention entropy change when restricting to the circuit?
4. Is the induction pattern preserved for induction heads in-circuit?
5. Which heads change most between base and circuit modes?
6. Do in-circuit vs out-of-circuit heads differ in their attention characteristics?

## Hypothesis Domain: R4-Circuit
- **H-R4c.1**: Circuit ablation preserves dominant head roles (induction, BOS sink)
- **H-R4c.2**: In-circuit heads maintain lower entropy (more specialized) than out-of-circuit heads
- **H-R4c.3**: Induction heads retain their induction cue attention under ablation
- **H-R4c.4**: Attention divergence (JS distance) is smaller for in-circuit heads
- **H-R4c.5**: Head role stability correlates with circuit membership

## Sections
1. Setup & Data Loading
2. Circuit Attention Patterns (positional fractions per head per band)
3. Circuit Head Role Classification
4. Circuit Attention Entropy
5. Induction Under Ablation
6. Base vs Circuit Comparison
7. In-Circuit vs Out-of-Circuit Head Analysis
8. Summary

## Data Sources
- Circuit activations: `outputs/extraction/circuit_activations/` (NPZ with `attn_pattern_predpos`)
- Base attention analysis: `outputs/attention/base/analysis/`
- Prune scores: ACDC circuit discovery outputs

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_PREDICTION_POS,
    MODEL_SOURCE_POS,
    MODEL_TARGET_POS,
    MODEL_REPEAT_POS,
    MODEL_INDUCTION_CUE_POS,
    SEQ_LEN_WITH_BOS,
    BOS_OFFSET,
    HEAD_ROLE_THRESHOLDS,
    get_domain_dirs,
)
from utils.data_loading import save_analysis, load_domain_csv
from utils.circuit_loading import (
    load_circuit_activations,
    load_base_and_circuit,
    load_prune_scores,
    get_circuit_mask,
)
from utils.attention import (
    compute_attention_entropy,
    classify_head_role,
    compute_positional_attention_stats,
)
from utils.plotting import setup_plotting, save_figure, plot_head_role_map

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jensenshannon

setup_plotting()

CIRCUIT_ANALYSIS, CIRCUIT_VIZ = get_domain_dirs("attention", "circuit")
COMP_ANALYSIS, COMP_VIZ = get_domain_dirs("attention", "comparison")
save_analysis_circuit = _partial(save_analysis, analysis_dir=CIRCUIT_ANALYSIS)
save_figure_circuit = _partial(save_figure, viz_dir=CIRCUIT_VIZ)
save_analysis_comp = _partial(save_analysis, analysis_dir=COMP_ANALYSIS)
save_figure_comp = _partial(save_figure, viz_dir=COMP_VIZ)

print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")
print(f"Circuit analysis: {CIRCUIT_ANALYSIS}")
print(f"Comparison analysis: {COMP_ANALYSIS}")
print(f"Prediction position (with BOS): {MODEL_PREDICTION_POS}")
print(f"Induction cue position (with BOS): {MODEL_INDUCTION_CUE_POS}")
print(f"Sequence length (with BOS): {SEQ_LEN_WITH_BOS}")
print(f"Head role thresholds: {HEAD_ROLE_THRESHOLDS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']
Circuit analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/circuit/analysis
Comparison analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/comparison/analysis
Prediction position (with BOS): 21
Induction cue position (with BOS): 5
Sequence length (with BOS): 22
Head role thresholds: {'induction_min': 0.15, 'bos_sink_min': 0.3, 'prev_token_min': 0.2, 'entropy_diffuse_min': 3.0, 'dominance_ratio': 1.5}


## 2. Circuit Attention Patterns

From the prediction position, compute the fraction of attention allocated to each
semantic region under circuit ablation: BOS (pos 0+BOS_OFFSET), source (MODEL_SOURCE_POS),
target (MODEL_TARGET_POS), distractors, repeat (MODEL_REPEAT_POS), and the
induction cue (MODEL_INDUCTION_CUE_POS). Per head, per band.

In [2]:
rows_pos = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    print(f"\nProcessing {model} ({n_layers}L x {n_heads}H)...")

    for band in BANDS:
        for draw in DRAWS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            attn = data["attn_pattern_predpos"]  # (N, n_layers, n_heads, seq_len)

            # Compute positional attention stats
            stats = compute_positional_attention_stats(attn)
            # Each stat is (N, n_layers, n_heads) -- take mean over examples

            for layer in range(n_layers):
                for head in range(n_heads):
                    record = {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "head": head,
                    }
                    for stat_name, stat_vals in stats.items():
                        record[stat_name] = float(stat_vals[:, layer, head].mean())
                        record[f"{stat_name}_std"] = float(
                            stat_vals[:, layer, head].std()
                        )
                    rows_pos.append(record)

df_circuit_pos = pd.DataFrame(rows_pos)
save_analysis_circuit(df_circuit_pos, "04c_circuit_positional_attention.csv")
print(f"\nCircuit positional attention records: {len(df_circuit_pos)}")
print(f"Columns: {list(df_circuit_pos.columns)}")
df_circuit_pos.head()


Processing pythia-70m (6L x 8H)...



Processing pythia-160m (12L x 12H)...



Processing pythia-410m (24L x 16H)...



Processing pythia-1b (16L x 8H)...



Processing pythia-1.4b (24L x 16H)...



Circuit positional attention records: 16320
Columns: ['model', 'draw', 'band', 'layer', 'head', 'bos_fraction', 'bos_fraction_std', 'self_fraction', 'self_fraction_std', 'prev_fraction', 'prev_fraction_std', 'source_fraction', 'source_fraction_std', 'target_fraction', 'target_fraction_std', 'repeat_fraction', 'repeat_fraction_std', 'induction_score', 'induction_score_std', 'distractor_fraction', 'distractor_fraction_std', 'entropy', 'entropy_std']


,model,draw,band,layer,head,bos_fraction,bos_fraction_std,self_fraction,self_fraction_std,prev_fraction,...,target_fraction,target_fraction_std,repeat_fraction,repeat_fraction_std,induction_score,induction_score_std,distractor_fraction,distractor_fraction_std,entropy,entropy_std
0,pythia-70m,draw_1,low,0,0,0.137045,0.111444,0.038538,0.023791,0.064796,...,0.031346,0.021726,0.307761,0.079753,0.015468,0.009740,0.416269,0.096677,3.948562,0.268941
1,pythia-70m,draw_1,low,0,1,0.000630,0.000892,0.226218,0.078603,0.396378,...,0.003913,0.003347,0.836745,0.067724,0.001263,0.001433,0.133536,0.055114,2.641586,0.395924
2,pythia-70m,draw_1,low,0,2,0.463540,0.206766,0.055215,0.051753,0.018947,...,0.020128,0.036725,0.149641,0.090641,0.080150,0.070740,0.223117,0.151559,2.608785,0.676062
3,pythia-70m,draw_1,low,0,3,0.012433,0.023351,0.063499,0.022089,0.000042,...,0.000977,0.005956,0.063914,0.022247,0.917411,0.046385,0.004422,0.020529,0.465097,0.203308
4,pythia-70m,draw_1,low,0,4,0.418906,0.253559,0.165076,0.171181,0.162111,...,0.005588,0.036820,0.456377,0.240055,0.001874,0.005348,0.097199,0.099390,2.009573,0.600033


In [3]:
# Visualization: mean attention to each region per layer (draw_1, averaged over heads)
regions = [
    "bos_fraction",
    "source_fraction",
    "target_fraction",
    "distractor_fraction",
    "repeat_fraction",
    "induction_score",
]
region_labels = [
    "BOS",
    "Source (S1-S5)",
    "Target (T)",
    "Distractors",
    "Repeat (S1-S5)",
    "Induction Cue",
]
region_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]

for model in MODELS:
    model_data = df_circuit_pos[
        (df_circuit_pos["model"] == model) & (df_circuit_pos["draw"] == "draw_1")
    ]
    if model_data.empty:
        continue

    n_bands = len(BANDS)
    fig, axes = plt.subplots(1, n_bands, figsize=(5 * n_bands, 5), sharey=True)
    if n_bands == 1:
        axes = [axes]

    for ax, band in zip(axes, BANDS):
        band_data = model_data[model_data["band"] == band]
        if band_data.empty:
            continue

        # Average over heads per layer
        layer_means = band_data.groupby("layer")[regions].mean()

        for region, label, color in zip(regions, region_labels, region_colors):
            if region in layer_means.columns:
                ax.plot(
                    layer_means.index,
                    layer_means[region],
                    label=label,
                    color=color,
                    marker="o",
                    markersize=3,
                )

        ax.set_xlabel("Layer")
        ax.set_title(BAND_NAMES.get(band, band))

    axes[0].set_ylabel("Mean Attention Fraction (Circuit)")
    axes[0].legend(fontsize=7, loc="upper left")
    fig.suptitle(f"Circuit Positional Attention by Region \u2014 {model}", y=1.02)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_04c_01_circuit_positional_attention_{model}.png")

## 3. Circuit Head Role Classification

Classify each head's role under circuit ablation using `classify_head_role()`.
Uses the same dominance-ratio criterion as the base analysis.

In [4]:
from collections import Counter

rows_roles = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    print(f"\n{model}:")

    # Combine all bands from draw_1 for overall classification
    all_band_attn = []
    for band in BANDS:
        try:
            data = load_circuit_activations(model, band, "draw_1")
            all_band_attn.append(data["attn_pattern_predpos"])
        except FileNotFoundError:
            pass

    if len(all_band_attn) == 0:
        print("  No circuit activations found, skipping.")
        continue

    combined_attn = np.concatenate(
        all_band_attn, axis=0
    )  # (N_total, n_layers, n_heads, 22)
    mean_attn = combined_attn.mean(axis=0)  # (n_layers, n_heads, 22)

    for layer in range(n_layers):
        for head in range(n_heads):
            head_attn = mean_attn[layer, head]  # (seq_len_with_bos,)

            # Compute positional stats for this single head
            stats = compute_positional_attention_stats(
                head_attn[np.newaxis, np.newaxis, np.newaxis, :]
            )
            head_stats = {k: float(v.item()) for k, v in stats.items()}

            role = classify_head_role(head_stats)

            rows_roles.append(
                {
                    "model": model,
                    "layer": layer,
                    "head": head,
                    "role": role,
                    **head_stats,
                }
            )

    # Print role distribution
    model_roles = [r["role"] for r in rows_roles if r["model"] == model]
    role_counts = Counter(model_roles)
    total = len(model_roles)
    for role_name, count in sorted(role_counts.items()):
        print(f"  {role_name}: {count}/{total} ({100 * count / total:.1f}%)")

df_circuit_roles = pd.DataFrame(rows_roles)
save_analysis_circuit(df_circuit_roles, "04c_circuit_head_roles.csv")
print(f"\nCircuit head role records: {len(df_circuit_roles)}")


pythia-70m:


  bos_sink: 17/48 (35.4%)
  diffuse: 17/48 (35.4%)
  induction: 3/48 (6.2%)
  previous_token: 11/48 (22.9%)

pythia-160m:


  bos_sink: 83/144 (57.6%)
  diffuse: 36/144 (25.0%)
  induction: 5/144 (3.5%)
  previous_token: 20/144 (13.9%)

pythia-410m:


  bos_sink: 241/384 (62.8%)
  diffuse: 110/384 (28.6%)
  induction: 8/384 (2.1%)
  previous_token: 25/384 (6.5%)

pythia-1b:


  bos_sink: 69/128 (53.9%)
  diffuse: 44/128 (34.4%)
  induction: 4/128 (3.1%)
  previous_token: 11/128 (8.6%)

pythia-1.4b:


  bos_sink: 259/384 (67.4%)
  diffuse: 100/384 (26.0%)
  induction: 7/384 (1.8%)
  previous_token: 18/384 (4.7%)

Circuit head role records: 1088


In [5]:
# Visualization: head role heatmap per model (circuit mode)
for model in MODELS:
    model_roles = df_circuit_roles[df_circuit_roles["model"] == model]
    if model_roles.empty:
        continue

    fig = plot_head_role_map(
        model_roles,
        model,
        title=f"Circuit Head Role Classification \u2014 {model}",
    )
    save_figure_circuit(fig, f"viz_04c_02_circuit_head_roles_{model}.png")

In [6]:
# Summary table: circuit role distribution per model
if not df_circuit_roles.empty:
    role_summary = (
        df_circuit_roles.groupby(["model", "role"]).size().unstack(fill_value=0)
    )
    role_summary["total"] = role_summary.sum(axis=1)
    for role in ["induction", "previous_token", "bos_sink", "diffuse"]:
        if role in role_summary.columns:
            role_summary[f"{role}_pct"] = (
                100 * role_summary[role] / role_summary["total"]
            ).round(1)

    save_analysis_circuit(
        role_summary.reset_index(), "04c_circuit_head_role_summary.csv"
    )
    print("Circuit Head Role Distribution per Model:")
    print(role_summary)

Circuit Head Role Distribution per Model:
role         bos_sink  diffuse  induction  previous_token  total  \
model                                                              
pythia-1.4b       259      100          7              18    384   
pythia-160m        83       36          5              20    144   
pythia-1b          69       44          4              11    128   
pythia-410m       241      110          8              25    384   
pythia-70m         17       17          3              11     48   

role         induction_pct  previous_token_pct  bos_sink_pct  diffuse_pct  
model                                                                      
pythia-1.4b            1.8                 4.7          67.4         26.0  
pythia-160m            3.5                13.9          57.6         25.0  
pythia-1b              3.1                 8.6          53.9         34.4  
pythia-410m            2.1                 6.5          62.8         28.6  
pythia-70m             6.

## 4. Circuit Attention Entropy

Compute attention entropy in **bits** (base=2) for each head under circuit ablation.
Lower entropy indicates more concentrated/specialized attention; higher entropy means diffuse.

In [7]:
rows_entropy = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    for draw in DRAWS:
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            attn = data["attn_pattern_predpos"]  # (N, n_layers, n_heads, seq_len)

            # Compute entropy per example, per layer, per head (bits)
            entropy = compute_attention_entropy(attn, base=2)  # (N, n_layers, n_heads)

            for layer in range(n_layers):
                for head in range(n_heads):
                    head_entropy = entropy[:, layer, head]
                    rows_entropy.append(
                        {
                            "model": model,
                            "draw": draw,
                            "band": band,
                            "layer": layer,
                            "head": head,
                            "entropy_mean": float(head_entropy.mean()),
                            "entropy_std": float(head_entropy.std()),
                            "entropy_median": float(np.median(head_entropy)),
                        }
                    )

df_circuit_entropy = pd.DataFrame(rows_entropy)
save_analysis_circuit(df_circuit_entropy, "04c_circuit_entropy.csv")
print(f"Circuit entropy records: {len(df_circuit_entropy)}")
print(
    f"Max entropy for {SEQ_LEN_WITH_BOS} positions = {np.log2(SEQ_LEN_WITH_BOS):.2f} bits"
)

Circuit entropy records: 16320
Max entropy for 22 positions = 4.46 bits


In [8]:
# Visualization: circuit entropy heatmap per model (head x layer, draw_1, avg over bands)
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    model_data = df_circuit_entropy[
        (df_circuit_entropy["model"] == model)
        & (df_circuit_entropy["draw"] == "draw_1")
    ]
    if model_data.empty:
        continue

    # Average over bands to get overall head characterization
    avg_data = (
        model_data.groupby(["layer", "head"])["entropy_mean"].mean().reset_index()
    )
    pivot = avg_data.pivot(index="head", columns="layer", values="entropy_mean")

    fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))
    sns.heatmap(
        pivot,
        annot=n_heads <= 16,
        fmt=".2f",
        cmap="viridis",
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"Circuit Attention Entropy (bits) \u2014 {model}")
    save_figure_circuit(fig, f"viz_04c_03_circuit_entropy_heatmap_{model}.png")

In [9]:
# Entropy distribution across layers (mean over heads, draw_1, per band)
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

for ax, model in zip(axes, MODELS):
    model_data = df_circuit_entropy[
        (df_circuit_entropy["model"] == model)
        & (df_circuit_entropy["draw"] == "draw_1")
    ]
    if model_data.empty:
        continue

    for band in BANDS:
        bd = model_data[model_data["band"] == band]
        if bd.empty:
            continue
        layer_means = bd.groupby("layer")["entropy_mean"].mean()
        ax.plot(
            layer_means.index,
            layer_means.values,
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )

    ax.set_xlabel("Layer")
    ax.set_title(model)
    ax.axhline(
        y=np.log2(SEQ_LEN_WITH_BOS),
        color="gray",
        linestyle="--",
        alpha=0.3,
        label="Max entropy",
    )

axes[0].set_ylabel("Mean Entropy (bits)")
axes[0].legend(fontsize=7)
fig.suptitle("Circuit Attention Entropy Across Layers (avg over heads)", y=1.02)
fig.tight_layout()
save_figure_circuit(fig, "viz_04c_04_circuit_entropy_by_layer.png")

## 5. Induction Under Ablation

Is the induction pattern preserved under circuit ablation? Check attention to
`MODEL_INDUCTION_CUE_POS` for heads classified as induction in either base or circuit mode.
Also examine whether non-induction heads gain induction-like patterns under ablation.

In [10]:
# Load base head role classifications
try:
    df_base_roles = load_domain_csv(
        "attention", "base", "04_head_role_classification.csv"
    )
    print(f"Base head roles loaded: {len(df_base_roles)} rows")
except FileNotFoundError:
    print("Base head role classification not found.")
    df_base_roles = pd.DataFrame()

# Identify induction heads from base and circuit classifications
induction_heads_base = {}
induction_heads_circuit = {}

if not df_base_roles.empty:
    for model in MODELS:
        base_ind = df_base_roles[
            (df_base_roles["model"] == model) & (df_base_roles["role"] == "induction")
        ]
        induction_heads_base[model] = list(
            zip(base_ind["layer"].values, base_ind["head"].values)
        )
        print(f"{model} base induction heads: {len(induction_heads_base[model])}")

if not df_circuit_roles.empty:
    for model in MODELS:
        circ_ind = df_circuit_roles[
            (df_circuit_roles["model"] == model)
            & (df_circuit_roles["role"] == "induction")
        ]
        induction_heads_circuit[model] = list(
            zip(circ_ind["layer"].values, circ_ind["head"].values)
        )
        print(f"{model} circuit induction heads: {len(induction_heads_circuit[model])}")

Base head roles loaded: 1088 rows
pythia-70m base induction heads: 3
pythia-160m base induction heads: 5
pythia-410m base induction heads: 8
pythia-1b base induction heads: 4
pythia-1.4b base induction heads: 7
pythia-70m circuit induction heads: 3
pythia-160m circuit induction heads: 5
pythia-410m circuit induction heads: 8
pythia-1b circuit induction heads: 4
pythia-1.4b circuit induction heads: 7


In [11]:
# For all heads that are induction in either base or circuit, compare their
# induction cue attention scores between the two modes
rows_induction = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Union of induction heads from base and circuit
    ind_base = set(induction_heads_base.get(model, []))
    ind_circuit = set(induction_heads_circuit.get(model, []))
    all_induction = ind_base | ind_circuit

    if not all_induction:
        # If no induction heads found, check top induction-scoring heads from circuit
        if not df_circuit_pos.empty:
            model_pos = df_circuit_pos[
                (df_circuit_pos["model"] == model)
                & (df_circuit_pos["draw"] == "draw_1")
            ]
            if not model_pos.empty:
                top_induction = (
                    model_pos.groupby(["layer", "head"])["induction_score"]
                    .mean()
                    .nlargest(5)
                )
                all_induction = set(
                    (int(idx[0]), int(idx[1])) for idx in top_induction.index
                )
                print(f"{model}: no classified induction heads; using top-5 by score")

    for band in BANDS:
        for draw in DRAWS:
            try:
                base_data, circuit_data = load_base_and_circuit(model, band, draw)
            except FileNotFoundError:
                continue

            base_attn = base_data["attn_pattern_predpos"]  # (N, L, H, S)
            circuit_attn = circuit_data["attn_pattern_predpos"]  # (N, L, H, S)

            for layer, head in all_induction:
                base_induction_score = float(
                    base_attn[:, layer, head, MODEL_INDUCTION_CUE_POS].mean()
                )
                circuit_induction_score = float(
                    circuit_attn[:, layer, head, MODEL_INDUCTION_CUE_POS].mean()
                )

                rows_induction.append(
                    {
                        "model": model,
                        "band": band,
                        "draw": draw,
                        "layer": layer,
                        "head": head,
                        "base_induction_score": base_induction_score,
                        "circuit_induction_score": circuit_induction_score,
                        "delta_induction": circuit_induction_score
                        - base_induction_score,
                        "in_base_induction": (layer, head) in ind_base,
                        "in_circuit_induction": (layer, head) in ind_circuit,
                    }
                )

df_induction_ablation = pd.DataFrame(rows_induction)
if not df_induction_ablation.empty:
    save_analysis_circuit(df_induction_ablation, "04c_induction_under_ablation.csv")
    print(f"Induction under ablation records: {len(df_induction_ablation)}")
    print(
        df_induction_ablation.groupby("model")[
            ["base_induction_score", "circuit_induction_score", "delta_induction"]
        ]
        .mean()
        .round(4)
    )
else:
    print("No induction analysis records (no circuit activations found).")

Induction under ablation records: 405
             base_induction_score  circuit_induction_score  delta_induction
model                                                                      
pythia-1.4b                0.7410                   0.7463           0.0053
pythia-160m                0.4983                   0.4986           0.0002
pythia-1b                  0.6457                   0.6536           0.0079
pythia-410m                0.6352                   0.6370           0.0018
pythia-70m                 0.6482                   0.6490           0.0008


In [12]:
# Visualization: base vs circuit induction score per model
if not df_induction_ablation.empty:
    for model in MODELS:
        model_data = df_induction_ablation[df_induction_ablation["model"] == model]
        if model_data.empty:
            continue

        # Average over draws for each band/head
        avg_data = (
            model_data.groupby(["band", "layer", "head"])
            .agg(
                {
                    "base_induction_score": "mean",
                    "circuit_induction_score": "mean",
                }
            )
            .reset_index()
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: scatter base vs circuit induction score
        ax = axes[0]
        for band in BANDS:
            bd = avg_data[avg_data["band"] == band]
            if bd.empty:
                continue
            ax.scatter(
                bd["base_induction_score"],
                bd["circuit_induction_score"],
                color=BAND_COLORS.get(band, "gray"),
                label=BAND_NAMES.get(band, band),
                alpha=0.7,
                s=30,
            )
        lims = [
            0,
            max(
                avg_data["base_induction_score"].max(),
                avg_data["circuit_induction_score"].max(),
            )
            * 1.1,
        ]
        ax.plot(lims, lims, "k--", alpha=0.3, label="y=x")
        ax.set_xlabel("Base Induction Score")
        ax.set_ylabel("Circuit Induction Score")
        ax.set_title("Induction Score: Base vs Circuit")
        ax.legend(fontsize=7)

        # Right: delta induction score by band
        ax = axes[1]
        band_deltas = model_data.groupby("band")["delta_induction"].mean()
        bands_present = [b for b in BANDS if b in band_deltas.index]
        colors = [BAND_COLORS.get(b, "gray") for b in bands_present]
        ax.bar(
            range(len(bands_present)),
            [band_deltas[b] for b in bands_present],
            color=colors,
        )
        ax.set_xticks(range(len(bands_present)))
        ax.set_xticklabels(
            [BAND_NAMES.get(b, b) for b in bands_present],
            rotation=45,
            ha="right",
            fontsize=8,
        )
        ax.set_ylabel("Delta Induction Score (Circuit - Base)")
        ax.set_title("Induction Score Change Under Ablation")
        ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

        fig.suptitle(f"Induction Under Ablation \u2014 {model}", y=1.02)
        fig.tight_layout()
        save_figure_circuit(fig, f"viz_04c_05_induction_ablation_{model}.png")
else:
    print("Skipping induction ablation visualization (no data).")

## 6. Base vs Circuit Comparison

Compare base and circuit attention patterns:
- Head role stability: count how many heads change classification
- Attention JS divergence between base and circuit per head
- Entropy change per head

In [13]:
# Load base head roles for comparison
try:
    df_base_roles = load_domain_csv(
        "attention", "base", "04_head_role_classification.csv"
    )
    print(f"Base head roles: {len(df_base_roles)} rows")
except FileNotFoundError:
    df_base_roles = pd.DataFrame()
    print("Base head roles not found, skipping comparison.")

Base head roles: 1088 rows


In [14]:
# Head role stability: base vs circuit
rows_stability = []

if not df_base_roles.empty and not df_circuit_roles.empty:
    for model in MODELS:
        base_m = df_base_roles[df_base_roles["model"] == model]
        circ_m = df_circuit_roles[df_circuit_roles["model"] == model]

        if base_m.empty or circ_m.empty:
            continue

        n_layers = MODEL_INFO[model]["n_layers"]
        n_heads = MODEL_INFO[model]["n_heads"]
        n_changed = 0
        n_total = 0

        for layer in range(n_layers):
            for head in range(n_heads):
                base_row = base_m[(base_m["layer"] == layer) & (base_m["head"] == head)]
                circ_row = circ_m[(circ_m["layer"] == layer) & (circ_m["head"] == head)]

                if base_row.empty or circ_row.empty:
                    continue

                base_role = base_row.iloc[0]["role"]
                circ_role = circ_row.iloc[0]["role"]
                changed = base_role != circ_role

                rows_stability.append(
                    {
                        "model": model,
                        "layer": layer,
                        "head": head,
                        "base_role": base_role,
                        "circuit_role": circ_role,
                        "role_changed": changed,
                    }
                )

                n_total += 1
                if changed:
                    n_changed += 1

        pct_changed = 100 * n_changed / n_total if n_total > 0 else 0
        print(f"{model}: {n_changed}/{n_total} heads changed role ({pct_changed:.1f}%)")

df_role_stability = pd.DataFrame(rows_stability)
if not df_role_stability.empty:
    save_analysis_comp(df_role_stability, "04c_role_stability_base_circuit.csv")
    print(f"\nRole stability records: {len(df_role_stability)}")
else:
    print("No role stability data available.")

pythia-70m: 0/48 heads changed role (0.0%)
pythia-160m: 1/144 heads changed role (0.7%)


pythia-410m: 2/384 heads changed role (0.5%)
pythia-1b: 5/128 heads changed role (3.9%)


pythia-1.4b: 10/384 heads changed role (2.6%)

Role stability records: 1088


In [15]:
# Visualization: role transition heatmap (base_role x circuit_role)
if not df_role_stability.empty:
    for model in MODELS:
        md = df_role_stability[df_role_stability["model"] == model]
        if md.empty:
            continue

        all_roles = ["induction", "previous_token", "bos_sink", "diffuse"]
        present_roles = [
            r
            for r in all_roles
            if r in md["base_role"].values or r in md["circuit_role"].values
        ]

        transition = pd.crosstab(
            md["base_role"], md["circuit_role"], dropna=False
        ).reindex(index=present_roles, columns=present_roles, fill_value=0)

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            transition,
            annot=True,
            fmt="d",
            cmap="Blues",
            square=True,
            linewidths=0,
            linecolor="none",
            ax=ax,
        )
        ax.set_xlabel("Circuit Role")
        ax.set_ylabel("Base Role")
        ax.set_title(f"Head Role Transitions: Base \u2192 Circuit \u2014 {model}")
        fig.tight_layout()
        save_figure_comp(fig, f"viz_04c_06_role_transitions_{model}.png")

In [16]:
# Attention JS divergence between base and circuit per head
rows_js = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    for band in BANDS:
        for draw in DRAWS:
            try:
                base_data, circuit_data = load_base_and_circuit(model, band, draw)
            except FileNotFoundError:
                continue

            base_attn = base_data["attn_pattern_predpos"]  # (N, L, H, S)
            circuit_attn = circuit_data["attn_pattern_predpos"]  # (N, L, H, S)

            # Mean attention pattern per head
            base_mean = base_attn.mean(axis=0)  # (L, H, S)
            circuit_mean = circuit_attn.mean(axis=0)  # (L, H, S)

            for layer in range(n_layers):
                for head in range(n_heads):
                    p = base_mean[layer, head]  # (S,)
                    q = circuit_mean[layer, head]  # (S,)

                    # Ensure valid probability distributions
                    p = np.clip(p, 1e-10, None)
                    q = np.clip(q, 1e-10, None)
                    p = p / p.sum()
                    q = q / q.sum()

                    js_dist = float(jensenshannon(p, q))

                    rows_js.append(
                        {
                            "model": model,
                            "band": band,
                            "draw": draw,
                            "layer": layer,
                            "head": head,
                            "js_divergence": js_dist,
                        }
                    )

df_js = pd.DataFrame(rows_js)
if not df_js.empty:
    save_analysis_comp(df_js, "04c_attention_js_divergence.csv")
    print(f"JS divergence records: {len(df_js)}")
    print("\nMean JS divergence per model:")
    print(df_js.groupby("model")["js_divergence"].mean().round(4))
else:
    print("No JS divergence data (no paired base/circuit activations found).")

<TMPDIR>/env/lib/python3.12/site-packages/scipy/spatial/distance.py:1290: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(js / 2.0)


JS divergence records: 16320

Mean JS divergence per model:
model
pythia-1.4b    0.0391
pythia-160m    0.0125
pythia-1b      0.0215
pythia-410m    0.0151
pythia-70m     0.0044
Name: js_divergence, dtype: float64


In [17]:
# Visualization: JS divergence heatmap per model (head x layer, averaged over bands/draws)
if not df_js.empty:
    for model in MODELS:
        md = df_js[df_js["model"] == model]
        if md.empty:
            continue

        n_layers = MODEL_INFO[model]["n_layers"]
        n_heads = MODEL_INFO[model]["n_heads"]

        avg = md.groupby(["layer", "head"])["js_divergence"].mean().reset_index()
        pivot = avg.pivot(index="head", columns="layer", values="js_divergence")

        fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))
        sns.heatmap(
            pivot,
            annot=False,
            cmap="YlOrRd",
            square=True,
            linewidths=0,
            linecolor="none",
            ax=ax,
        )
        ax.set_xlabel("Layer")
        ax.set_ylabel("Head")
        ax.set_title(f"Attention JS Divergence (Base vs Circuit) \u2014 {model}")
        fig.tight_layout()
        save_figure_comp(fig, f"viz_04c_07_js_divergence_{model}.png")

In [18]:
# Entropy change per head: base vs circuit
try:
    df_base_entropy = load_domain_csv("attention", "base", "04_attention_entropy.csv")
    print(f"Base entropy loaded: {len(df_base_entropy)} rows")
except FileNotFoundError:
    df_base_entropy = pd.DataFrame()
    print("Base entropy not found.")

rows_entropy_change = []

if not df_base_entropy.empty and not df_circuit_entropy.empty:
    for model in MODELS:
        for draw in DRAWS:
            for band in BANDS:
                base_e = df_base_entropy[
                    (df_base_entropy["model"] == model)
                    & (df_base_entropy["draw"] == draw)
                    & (df_base_entropy["band"] == band)
                ]
                circ_e = df_circuit_entropy[
                    (df_circuit_entropy["model"] == model)
                    & (df_circuit_entropy["draw"] == draw)
                    & (df_circuit_entropy["band"] == band)
                ]

                if base_e.empty or circ_e.empty:
                    continue

                merged = base_e.merge(
                    circ_e,
                    on=["model", "draw", "band", "layer", "head"],
                    suffixes=("_base", "_circuit"),
                )

                for _, row in merged.iterrows():
                    rows_entropy_change.append(
                        {
                            "model": row["model"],
                            "draw": row["draw"],
                            "band": row["band"],
                            "layer": int(row["layer"]),
                            "head": int(row["head"]),
                            "entropy_base": float(row["entropy_mean_base"]),
                            "entropy_circuit": float(row["entropy_mean_circuit"]),
                            "delta_entropy": float(
                                row["entropy_mean_circuit"] - row["entropy_mean_base"]
                            ),
                        }
                    )

df_entropy_change = pd.DataFrame(rows_entropy_change)
if not df_entropy_change.empty:
    save_analysis_comp(df_entropy_change, "04c_entropy_change_base_circuit.csv")
    print(f"Entropy change records: {len(df_entropy_change)}")
    print("\nMean entropy change per model (bits):")
    print(
        df_entropy_change.groupby("model")[
            ["entropy_base", "entropy_circuit", "delta_entropy"]
        ]
        .mean()
        .round(4)
    )
else:
    print("No entropy change data available.")

Base entropy loaded: 16320 rows


Entropy change records: 16320

Mean entropy change per model (bits):
             entropy_base  entropy_circuit  delta_entropy
model                                                    
pythia-1.4b        1.9531           1.9363        -0.0168
pythia-160m        1.6614           1.6628         0.0014
pythia-1b          2.5692           2.5712         0.0020
pythia-410m        2.0980           2.1017         0.0036
pythia-70m         1.8596           1.8621         0.0025


In [19]:
# Visualization: entropy change heatmap per model
if not df_entropy_change.empty:
    for model in MODELS:
        md = df_entropy_change[df_entropy_change["model"] == model]
        if md.empty:
            continue

        n_layers = MODEL_INFO[model]["n_layers"]
        n_heads = MODEL_INFO[model]["n_heads"]

        avg = md.groupby(["layer", "head"])["delta_entropy"].mean().reset_index()
        pivot = avg.pivot(index="head", columns="layer", values="delta_entropy")

        fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))
        vmax = max(abs(pivot.values.min()), abs(pivot.values.max()))
        sns.heatmap(
            pivot,
            annot=n_heads <= 16,
            fmt=".2f",
            cmap="RdBu_r",
            square=True,
            linewidths=0,
            linecolor="none",
            center=0,
            vmin=-vmax,
            vmax=vmax,
            ax=ax,
        )
        ax.set_xlabel("Layer")
        ax.set_ylabel("Head")
        ax.set_title(f"Entropy Change (Circuit - Base, bits) \u2014 {model}")
        fig.tight_layout()
        save_figure_comp(fig, f"viz_04c_08_entropy_change_{model}.png")

In [20]:
# Overlay: base vs circuit entropy trajectory per model
if not df_base_entropy.empty and not df_circuit_entropy.empty:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        # Base (solid)
        base_m = df_base_entropy[
            (df_base_entropy["model"] == model) & (df_base_entropy["draw"] == "draw_1")
        ]
        if not base_m.empty:
            layer_means = base_m.groupby("layer")["entropy_mean"].mean()
            ax.plot(
                layer_means.index,
                layer_means.values,
                color="steelblue",
                label="Base",
                linewidth=2,
                marker="o",
                ms=4,
            )

        # Circuit (dashed)
        circ_m = df_circuit_entropy[
            (df_circuit_entropy["model"] == model)
            & (df_circuit_entropy["draw"] == "draw_1")
        ]
        if not circ_m.empty:
            layer_means = circ_m.groupby("layer")["entropy_mean"].mean()
            ax.plot(
                layer_means.index,
                layer_means.values,
                color="coral",
                label="Circuit",
                linewidth=2,
                linestyle="--",
                marker="s",
                ms=4,
            )

        ax.axhline(
            y=np.log2(SEQ_LEN_WITH_BOS),
            color="gray",
            linestyle=":",
            alpha=0.3,
            label="Max entropy",
        )
        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean Entropy (bits)")
        ax.set_title(f"Base vs Circuit Entropy \u2014 {model}")
        ax.legend()
        fig.tight_layout()
        save_figure_comp(fig, f"viz_04c_09_entropy_overlay_{model}.png")

## 7. In-Circuit vs Out-of-Circuit Head Analysis

Use `get_circuit_mask()` to identify which heads are in-circuit (have at least one
in-circuit edge). Compare attention characteristics of in-circuit vs out-of-circuit heads.

In [21]:
# Build aggregated circuit mask per model (union over bands/draws)
circuit_masks = {}

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Aggregate: a head is "in-circuit" if it appears in ANY band/draw circuit
    agg_head_mask = np.zeros((n_layers, n_heads), dtype=bool)
    n_circuits_found = 0

    for band in BANDS:
        for draw in DRAWS:
            try:
                ps = load_prune_scores(model, band, draw)
                cm = get_circuit_mask(ps, n_layers, n_heads)
                agg_head_mask |= cm["head_mask"]
                n_circuits_found += 1
            except FileNotFoundError:
                pass

    circuit_masks[model] = agg_head_mask
    n_in = int(agg_head_mask.sum())
    n_out = n_layers * n_heads - n_in
    print(
        f"{model}: {n_in} in-circuit, {n_out} out-of-circuit heads "
        f"({n_circuits_found} circuits found)"
    )

pythia-70m: 45 in-circuit, 3 out-of-circuit heads (15 circuits found)
pythia-160m: 127 in-circuit, 17 out-of-circuit heads (15 circuits found)


pythia-410m: 305 in-circuit, 79 out-of-circuit heads (15 circuits found)
pythia-1b: 113 in-circuit, 15 out-of-circuit heads (15 circuits found)


pythia-1.4b: 284 in-circuit, 100 out-of-circuit heads (15 circuits found)


In [22]:
# Compare attention characteristics: in-circuit vs out-of-circuit heads
rows_in_out = []

for model in MODELS:
    head_mask = circuit_masks.get(model)
    if head_mask is None:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Use circuit positional attention stats (draw_1)
    model_pos = df_circuit_pos[
        (df_circuit_pos["model"] == model) & (df_circuit_pos["draw"] == "draw_1")
    ]
    if model_pos.empty:
        continue

    # Use circuit entropy (draw_1)
    model_ent = df_circuit_entropy[
        (df_circuit_entropy["model"] == model)
        & (df_circuit_entropy["draw"] == "draw_1")
    ]

    # Classify each head
    for layer in range(n_layers):
        for head in range(n_heads):
            in_circuit = bool(head_mask[layer, head])

            # Positional stats (averaged over bands)
            head_pos = model_pos[
                (model_pos["layer"] == layer) & (model_pos["head"] == head)
            ]
            head_ent = model_ent[
                (model_ent["layer"] == layer) & (model_ent["head"] == head)
            ]

            if head_pos.empty:
                continue

            record = {
                "model": model,
                "layer": layer,
                "head": head,
                "in_circuit": in_circuit,
                "induction_score": head_pos["induction_score"].mean(),
                "bos_fraction": head_pos["bos_fraction"].mean(),
                "source_fraction": head_pos["source_fraction"].mean(),
                "target_fraction": head_pos["target_fraction"].mean(),
                "repeat_fraction": head_pos["repeat_fraction"].mean(),
            }

            if not head_ent.empty:
                record["entropy_mean"] = head_ent["entropy_mean"].mean()

            # Circuit role
            circ_role_row = df_circuit_roles[
                (df_circuit_roles["model"] == model)
                & (df_circuit_roles["layer"] == layer)
                & (df_circuit_roles["head"] == head)
            ]
            if not circ_role_row.empty:
                record["circuit_role"] = circ_role_row.iloc[0]["role"]

            # JS divergence (averaged over bands/draws)
            if not df_js.empty:
                head_js = df_js[
                    (df_js["model"] == model)
                    & (df_js["layer"] == layer)
                    & (df_js["head"] == head)
                ]
                if not head_js.empty:
                    record["mean_js_divergence"] = head_js["js_divergence"].mean()

            rows_in_out.append(record)

df_in_out = pd.DataFrame(rows_in_out)
if not df_in_out.empty:
    save_analysis_comp(df_in_out, "04c_in_vs_out_circuit_heads.csv")
    print(f"In/out circuit head records: {len(df_in_out)}")

    # Summary statistics
    stat_cols = ["induction_score", "bos_fraction", "entropy_mean"]
    available_stat_cols = [c for c in stat_cols if c in df_in_out.columns]
    if available_stat_cols:
        summary = (
            df_in_out.groupby(["model", "in_circuit"])[available_stat_cols]
            .mean()
            .round(4)
        )
        print("\nMean stats by circuit membership:")
        print(summary)
else:
    print("No in/out circuit analysis data available.")

In/out circuit head records: 1088

Mean stats by circuit membership:
                        induction_score  bos_fraction  entropy_mean
model       in_circuit                                             
pythia-1.4b False                0.0086        0.7723        1.0379
            True                 0.0361        0.3980        2.2488
pythia-160m False                0.0052        0.8467        0.5627
            True                 0.0362        0.3146        1.8062
pythia-1b   False                0.0151        0.6579        2.0559
            True                 0.0430        0.3142        2.6427
pythia-410m False                0.0134        0.6910        1.4077
            True                 0.0387        0.4051        2.2824
pythia-70m  False                0.0179        0.6932        1.1994
            True                 0.0550        0.2626        1.9035


In [23]:
# Visualization: in-circuit vs out-of-circuit heads comparison
if not df_in_out.empty:
    metrics_to_compare = [
        ("induction_score", "Induction Score"),
        ("bos_fraction", "BOS Fraction"),
        ("entropy_mean", "Entropy (bits)"),
    ]
    available_metrics = [
        (m, l) for m, l in metrics_to_compare if m in df_in_out.columns
    ]

    if available_metrics:
        for model in MODELS:
            md = df_in_out[df_in_out["model"] == model]
            if md.empty or md["in_circuit"].nunique() < 2:
                continue

            n_metrics = len(available_metrics)
            fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))
            if n_metrics == 1:
                axes = [axes]

            for ax, (metric, label) in zip(axes, available_metrics):
                in_vals = md[md["in_circuit"]][metric].dropna()
                out_vals = md[~md["in_circuit"]][metric].dropna()

                parts = ax.violinplot(
                    [in_vals.values, out_vals.values],
                    positions=[0, 1],
                    showmeans=True,
                    showmedians=True,
                )
                ax.set_xticks([0, 1])
                ax.set_xticklabels(["In-Circuit", "Out-of-Circuit"])
                ax.set_ylabel(label)
                ax.set_title(label)

            fig.suptitle(f"In-Circuit vs Out-of-Circuit Heads \u2014 {model}", y=1.02)
            fig.tight_layout()
            save_figure_comp(fig, f"viz_04c_10_in_out_circuit_{model}.png")

In [24]:
# Visualization: JS divergence by circuit membership
if not df_in_out.empty and "mean_js_divergence" in df_in_out.columns:
    for model in MODELS:
        md = df_in_out[df_in_out["model"] == model]
        if md.empty or md["in_circuit"].nunique() < 2:
            continue

        fig, ax = plt.subplots(figsize=(8, 5))

        in_js = md[md["in_circuit"]]["mean_js_divergence"].dropna()
        out_js = md[~md["in_circuit"]]["mean_js_divergence"].dropna()

        if len(in_js) > 0 and len(out_js) > 0:
            bp = ax.boxplot(
                [in_js.values, out_js.values],
                labels=["In-Circuit", "Out-of-Circuit"],
                patch_artist=True,
            )
            bp["boxes"][0].set_facecolor("#2ca02c")
            bp["boxes"][1].set_facecolor("#d62728")
            for box in bp["boxes"]:
                box.set_alpha(0.6)

            ax.set_ylabel("Mean JS Divergence (Base vs Circuit)")
            ax.set_title(f"Attention Divergence by Circuit Membership \u2014 {model}")
            fig.tight_layout()
            save_figure_comp(fig, f"viz_04c_11_js_by_membership_{model}.png")
        else:
            plt.close(fig)

<TMPDIR>/ipykernel_400829/478435768.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
<TMPDIR>/ipykernel_400829/478435768.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


<TMPDIR>/ipykernel_400829/478435768.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
<TMPDIR>/ipykernel_400829/478435768.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


<TMPDIR>/ipykernel_400829/478435768.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


In [25]:
# Visualization: circuit membership heatmap overlaid with role
for model in MODELS:
    head_mask = circuit_masks.get(model)
    if head_mask is None or not head_mask.any():
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))

    # head_mask shape is (n_layers, n_heads): transpose to (n_heads, n_layers)
    # so rows = heads (Y-axis), columns = layers (X-axis), matching other heatmaps
    mask_display = head_mask.T.astype(int)  # (n_heads, n_layers)

    # Binary heatmap: in-circuit (1) vs out (0)
    sns.heatmap(
        mask_display,
        annot=False,
        cmap="Greens",
        square=True,
        linewidths=0,
        linecolor="none",
        vmin=0,
        vmax=1,
        xticklabels=list(range(n_layers)),
        yticklabels=list(range(n_heads)),
        ax=ax,
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"Circuit Membership (1=in-circuit) \u2014 {model}")
    fig.tight_layout()
    save_figure_comp(fig, f"viz_04c_12_circuit_membership_{model}.png")

## 8. Summary

Consolidate circuit attention analysis results.

In [26]:
# Master summary per model
rows_summary = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    n_total = n_layers * n_heads

    # Circuit roles
    model_roles = df_circuit_roles[df_circuit_roles["model"] == model]
    if model_roles.empty:
        continue

    role_counts = model_roles["role"].value_counts()
    n_induction = int(role_counts.get("induction", 0))
    n_diffuse = int(role_counts.get("diffuse", 0))

    # Circuit entropy
    model_ent = df_circuit_entropy[
        (df_circuit_entropy["model"] == model)
        & (df_circuit_entropy["draw"] == "draw_1")
    ]
    mean_entropy = model_ent["entropy_mean"].mean() if not model_ent.empty else np.nan

    # Circuit positional stats
    model_pos = df_circuit_pos[
        (df_circuit_pos["model"] == model) & (df_circuit_pos["draw"] == "draw_1")
    ]
    mean_induction = (
        model_pos["induction_score"].mean() if not model_pos.empty else np.nan
    )
    mean_bos = model_pos["bos_fraction"].mean() if not model_pos.empty else np.nan

    # Role stability
    model_stab = (
        df_role_stability[df_role_stability["model"] == model]
        if not df_role_stability.empty
        else pd.DataFrame()
    )
    pct_role_changed = (
        100 * model_stab["role_changed"].mean() if not model_stab.empty else np.nan
    )

    # JS divergence
    model_js = df_js[df_js["model"] == model] if not df_js.empty else pd.DataFrame()
    mean_js = model_js["js_divergence"].mean() if not model_js.empty else np.nan

    # Circuit membership
    head_mask = circuit_masks.get(model)
    n_in_circuit = int(head_mask.sum()) if head_mask is not None else 0

    rows_summary.append(
        {
            "model": model,
            "model_capacity": MODEL_CAPACITY.get(model, 0),
            "n_total_heads": n_total,
            "n_in_circuit_heads": n_in_circuit,
            "pct_in_circuit": 100 * n_in_circuit / n_total if n_total > 0 else 0,
            "n_induction_circuit": n_induction,
            "n_diffuse_circuit": n_diffuse,
            "pct_specialized_circuit": 100 * (1 - n_diffuse / n_total)
            if n_total > 0
            else 0,
            "mean_entropy_circuit": mean_entropy,
            "mean_induction_score_circuit": mean_induction,
            "mean_bos_fraction_circuit": mean_bos,
            "pct_role_changed": pct_role_changed,
            "mean_js_divergence": mean_js,
        }
    )

df_summary = pd.DataFrame(rows_summary)
save_analysis_circuit(df_summary, "04c_circuit_attention_summary.csv")
print("Circuit Attention Summary:")
print(df_summary.to_string(index=False))

Circuit Attention Summary:
      model  model_capacity  n_total_heads  n_in_circuit_heads  pct_in_circuit  n_induction_circuit  n_diffuse_circuit  pct_specialized_circuit  mean_entropy_circuit  mean_induction_score_circuit  mean_bos_fraction_circuit  pct_role_changed  mean_js_divergence
 pythia-70m              70             48                  45       93.750000                    3                 17                64.583333              1.859463                      0.052727                   0.289541          0.000000            0.004442
pythia-160m             160            144                 127       88.194444                    5                 36                75.000000              1.659398                      0.032569                   0.377403          0.694444            0.012469
pythia-410m             410            384                 305       79.427083                    8                110                71.354167              2.102455                      0.0

In [27]:
# Cross-model scaling: circuit attention metrics vs model size
if not df_summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    x = df_summary["model_capacity"]

    axes[0].plot(x, df_summary["pct_specialized_circuit"], "o-", color="coral")
    axes[0].set_title("Circuit % Specialized Heads")
    axes[0].set_ylabel("% Specialized")

    axes[1].plot(x, df_summary["mean_entropy_circuit"], "o-", color="coral")
    axes[1].set_title("Circuit Mean Entropy (bits)")
    axes[1].set_ylabel("Entropy (bits)")

    if "pct_role_changed" in df_summary.columns:
        axes[2].plot(x, df_summary["pct_role_changed"], "o-", color="coral")
        axes[2].set_title("% Heads with Role Change")
        axes[2].set_ylabel("% Changed")

    for ax in axes:
        ax.set_xlabel("Model Capacity (M)")
        ax.set_xscale("log")

    fig.suptitle("Circuit Attention Metrics vs Model Size", y=1.02)
    fig.tight_layout()
    save_figure_circuit(fig, "viz_04c_13_cross_model_scaling.png")

In [28]:
# Per-band summary: circuit attention metrics by band
if not df_circuit_pos.empty:
    df_band_summary = (
        df_circuit_pos.groupby(["model", "band"])
        .agg(
            mean_induction_score=("induction_score", "mean"),
            mean_bos_fraction=("bos_fraction", "mean"),
            mean_target_fraction=("target_fraction", "mean"),
        )
        .reset_index()
    )
    save_analysis_circuit(df_band_summary, "04c_circuit_per_band_attention.csv")

    # Heatmap: model x band induction score
    pivot = df_band_summary.pivot(
        index="model", columns="band", values="mean_induction_score"
    )
    pivot = pivot.reindex(index=MODELS, columns=BANDS)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".4f",
        cmap="YlOrRd",
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_title("Circuit Mean Induction Score (Model x Band)")
    fig.tight_layout()
    save_figure_circuit(fig, "viz_04c_14_circuit_induction_heatmap.png")

In [29]:
print("\n" + "=" * 70)
print("NOTEBOOK 04c: ATTENTION CIRCUIT ANALYSIS COMPLETE")
print("=" * 70)

print(f"\nCircuit CSVs in: {CIRCUIT_ANALYSIS}")
for f in sorted(CIRCUIT_ANALYSIS.glob("04c_*")):
    print(f"  {f.name}")

print(f"\nComparison CSVs in: {COMP_ANALYSIS}")
for f in sorted(COMP_ANALYSIS.glob("04c_*")):
    print(f"  {f.name}")

print(f"\nCircuit figures in: {CIRCUIT_VIZ}")
for f in sorted(CIRCUIT_VIZ.glob("viz_04c_*")):
    print(f"  {f.name}")

print(f"\nComparison figures in: {COMP_VIZ}")
for f in sorted(COMP_VIZ.glob("viz_04c_*")):
    print(f"  {f.name}")

print("\n--- Key Findings ---")
for _, row in df_summary.iterrows():
    model = row["model"]
    print(f"\n{model}:")
    print(f"  Total heads: {int(row['n_total_heads'])}")
    print(
        f"  In-circuit heads: {int(row['n_in_circuit_heads'])} ({row['pct_in_circuit']:.1f}%)"
    )
    print(f"  Induction heads (circuit): {int(row['n_induction_circuit'])}")
    print(
        f"  Specialized (non-diffuse, circuit): {row['pct_specialized_circuit']:.1f}%"
    )
    print(f"  Mean entropy (circuit): {row['mean_entropy_circuit']:.2f} bits")
    if not np.isnan(row.get("pct_role_changed", np.nan)):
        print(f"  Heads with role change: {row['pct_role_changed']:.1f}%")
    if not np.isnan(row.get("mean_js_divergence", np.nan)):
        print(
            f"  Mean JS divergence (base vs circuit): {row['mean_js_divergence']:.4f}"
        )


NOTEBOOK 04c: ATTENTION CIRCUIT ANALYSIS COMPLETE

Circuit CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/circuit/analysis
  04c_circuit_attention_summary.csv
  04c_circuit_entropy.csv
  04c_circuit_head_role_summary.csv
  04c_circuit_head_roles.csv
  04c_circuit_per_band_attention.csv
  04c_circuit_positional_attention.csv
  04c_induction_under_ablation.csv

Comparison CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/comparison/analysis
  04c_attention_js_divergence.csv
  04c_entropy_change_base_circuit.csv
  04c_in_vs_out_circuit_heads.csv
  04c_role_stability_base_circuit.csv

Circuit figures in: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/circuit/viz
  viz_04c_01_circuit_positional_attention_pythia-1.4b.png
  viz_04c_01_circuit_positional_attention_pythia-160m.png
  viz_04c_01_circuit_positional_attention_pythia-1b.png
  viz_04c_01_circuit_positional_attention_pythia-410m.png
  viz_04c_01_circuit_positional_a